# Numerical simulation of point-to-plane corona discharge using a Monte Carlo method

based on "Numerical simulation of point-to-plane corona discharge using a Monte Carlo method" (2010) by Wook Hee Koh and In-Ho Park, investigates the transient dynamical properties and microscopic mechanisms of charged particles during a negative corona discharge in nitrogen gas

# 1. PHYSICAL CONSTANTS & SYSTEM PARAMETERS

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Fixed Random Seed for Reproducibility
np.random.seed(42)

e = 1.602e-19           # Elementary charge (C)
m_e = 9.109e-31          # Electron mass (kg)
m_i = 28.0134 * 1.66e-27 # Positive N2+ ion mass (kg)
eps0 = 8.854e-12         # Vacuum permittivity (F/m)
kB = 1.3806e-23          # Boltzmann constant (J/K)
eV_to_J = 1.602e-19      # Energy conversion (1 eV in Joules)

# Geometry and Discharge Parameters
d = 5.0e-3               # Inter-electrode gap distance: 0.5 cm
r_c = 1.0e-4             # Cathode tip radius: 0.01 cm
r_d = 2.0e-3             # Discharge cylinder radius: 0.2 cm
V0 = -3000.0             # Applied negative potential: -3 kV
P_gas = 13.3e3           # Nitrogen gas pressure: 13.3 kPa
T_gas = 300.0            # Temperature in Kelvin
N_gas = P_gas / (kB * T_gas)  # Neutral gas density (~3.21 x 10^24 m^-3)

# Spatial Grid
Nx = 100
x_grid = np.linspace(0, d, Nx)
x_cm = x_grid * 100      # Grid in cm for plotting
dx = d / (Nx - 1)


# 2. ANALYTIC LAPLACIAN FIELD E_L(x)

In [2]:
def calc_E_L(x):
    x_c = np.clip(x, 1e-6, d - 1e-6)
    denom = (d * (2 * x_c + r_c) - x_c**2) * np.log(4 * d / r_c)
    return (2 * V0 * d) / denom

E_L = calc_E_L(x_grid)

# 3. SPACE CHARGE FIELD E_S(x) DISK METHOD (Poisson's field)

In [3]:
def calc_E_S(rho_net, x_grid):
    E_S = np.zeros_like(x_grid)
    for i, x in enumerate(x_grid):
        # Integral across left sub-volume (-x to 0)
        x_left = x_grid[:i+1] - x
        kernel_left = -1.0 - x_left / np.sqrt(x_left**2 + r_d**2)
        int_left = np.trapz(rho_net[:i+1] * kernel_left, x_grid[:i+1]) if i > 0 else 0.0

        # Integral across right sub-volume (0 to d-x)
        x_right = x_grid[i:] - x
        kernel_right = 1.0 - x_right / np.sqrt(x_right**2 + r_d**2)
        int_right = np.trapz(rho_net[i:] * kernel_right, x_grid[i:]) if i < Nx-1 else 0.0

        E_S[i] = (1.0 / (2.0 * eps0)) * (int_left + int_right)
    return E_S

# 4. CROSS SECTION FUNCTIONS FOR N2 GAS

In [4]:
def cross_sections(energy_eV):
    e_ev = np.maximum(0.0, energy_eV)

    # Elastic cross section
    sigma_el = np.where(e_ev > 0.1, 1.2e-19 / (1.0 + 0.05 * e_ev), 5.0e-20)

    # Excitation cross section (threshold ~ 6 eV)
    sigma_ex = np.where(e_ev > 6.0, 3.0e-20 * (e_ev - 6.0) / (10.0 + e_ev**1.2), 0.0)

    # Ionization cross section (threshold ~ 15.6 eV)
    diff_ion = np.maximum(0.0, e_ev - 15.6)
    sigma_ion = np.where(e_ev > 15.6, 2.5e-20 * diff_ion / (20.0 + diff_ion**1.1), 0.0)

    return sigma_el, sigma_ex, sigma_ion